In [ ]:
!git clone https://github.com/dcbisson23/CISC839_A2
!git clone https://github.com/taoyds/test-suite-sql-eval

In [1]:
from transformers import set_seed
set_seed(20230334)

C:\Users\Donovan Bisson\AppData\Local\Python\pythoncore-3.10-64\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# This is mostly pulled from A1's py file, for ease of use.

# Kaggle
# TABLES_PATH = "/kaggle/input/datasets/dcbisson23/spider-1-0/spider/tables.json"
# QUESTIONS_PATH = "/kaggle/input/datasets/dcbisson23/spider-1-0/spider/dev.json"
# GOLD_PATH = "/kaggle/input/datasets/dcbisson23/spider-1-0/spider/dev_gold.sql"
# SPIDER_DB_PATH = "/kaggle/input/datasets/dcbisson23/spider-1-0/spider/database"
# EXAMPLES_PATH = "/kaggle/input/datasets/dcbisson23/spider-1-0/spider/train_spider.json"
# DATABASE_PATH = "/kaggle/input/datasets/dcbisson23/test-suite-sql-eval-database/database"
# DATA_ROOT = "/kaggle/working/CISC839_A2/data"

# Local
TABLES_PATH = "spider/tables.json"
QUESTIONS_PATH = "spider/dev.json"
GOLD_PATH = "spider/dev_gold.sql"
SPIDER_DB_PATH = "spider/database"
EXAMPLES_PATH = "spider/train_spider.json"
DATABASE_PATH = "test-suite-sql-eval/database"
DATA_ROOT = "data"

GEN_CONFIG_FILENAME = "generation_config.json"
PROMPTS_FILENAME = "formatted_prompts.jsonl"
RESPONSES_FILENAME = "responses.jsonl"
PREDICTIONS_FILENAME = "predictions.txt"
GOLD_FILENAME = "gold.txt"
AGENT_RESULTS_FILENAME = "agent_results.jsonl"

SAMPLE_RATE = 0.125
# Offset for the sampling of dev.json. Set to 0 for the set used in A1.
# OFFSET = 0
OFFSET = 1

# This is the system context used for prompting.
# NOTE: Modified from A1, removed clause restricting use of aliasing in query (found many gold examples used aliasing where A1 models did not).
SYS_CONTEXT = ("""You are an expert SQLite developer, tasked with writing a valid SQL query to answer a user question using a given SQL schema.
The user will provide the following information:
    1. The database schema as a standard SQLite DDL script, and
    3. The question to be answered by querying this database.
Your response must adhere to the following guidelines:
    1. (CRITICAL) You must provide ONLY a valid SQLite query to answer the provided question, and nothing else.
    2. Use ONLY the tables and columns provided in the schema.
    3. Do not use aliasing (x AS y) unless strictly necessary.
""")

# This is the model being used for A2.
MODEL = {
    "model": "Qwen/Qwen2.5-Coder-3B-Instruct"
         }
SENTENCE_TRANSFORMER = "sentence-transformers/all-MiniLM-L6-v2"
MAX_STEPS = 100

In [ ]:
import os
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(f"{DATA_ROOT}/{MODEL['model']}", exist_ok=True)

In [ ]:
from CISC839_A2.scripts.utils import load_tokenizer, unload_model
from CISC839_A2.scripts.generate_prompts import SpiderPromptBuilder

builder = SpiderPromptBuilder(TABLES_PATH, QUESTIONS_PATH, sys_context=SYS_CONTEXT)

tokenizer = load_tokenizer(MODEL)

prompts_path = f"{DATA_ROOT}/{MODEL['model']}/{PROMPTS_FILENAME}"

builder.generate_prompts(prompts_path, tokenizer, sample_rate=SAMPLE_RATE, offset=OFFSET)

unload_model(tokenizer)

In [ ]:
from CISC839_A2.scripts.utils import load_model, unload_model
from CISC839_A2.scripts.get_predictions import LLMPrompter

print("Loading model "+MODEL['model'])
model = load_model(MODEL)
tokenizer = load_tokenizer(MODEL)

gen_path = f"{DATA_ROOT}/{MODEL['model']}/{GEN_CONFIG_FILENAME}"
prompts_path = f"{DATA_ROOT}/{MODEL['model']}/{PROMPTS_FILENAME}"
responses_path = f"{DATA_ROOT}/{MODEL['model']}/{RESPONSES_FILENAME}"

prompter = LLMPrompter(model, tokenizer)
prompter.load_gen_config(gen_path)
prompter.run_prompt_pipeline(prompts_path, responses_path)

unload_model(model)
unload_model(tokenizer)

In [ ]:
import json
from CISC839_A2.scripts.utils import load_tokenizer, get_data_from_response
responses_path = f"{DATA_ROOT}/{MODEL['model']}/{RESPONSES_FILENAME}"
predictions_path = f"{DATA_ROOT}/{MODEL['model']}/{PREDICTIONS_FILENAME}"
gold_path = f"{DATA_ROOT}/{MODEL['model']}/{GOLD_FILENAME}"

with open(responses_path) as f:
    responses = f.readlines()
with open(GOLD_PATH) as f:
    gold_sql = f.readlines()


predictions = []
golds = []
reasoning_traces = []

for response_json in responses:
    response_dict = json.loads(response_json)
    id = response_dict["id"]
    db_id = response_dict["db_id"]
    response = response_dict["response"]
    response_data = get_data_from_response(response)
    pred = response_data["prediction"]
    gold = gold_sql[id]
    predictions.append(pred)
    # To match needed formatting for evaluation script.
    golds.append(gold)

with open(gold_path, "w", encoding="utf-8") as f_out:
    f_out.write("".join(golds))
with open(predictions_path, "w", encoding="utf-8") as f_out:
    f_out.write("\n".join(predictions))

In [ ]:
import os
import sys
cwd = os.getcwd()
sys.path.insert(0, f"{cwd}/test-suite-sql-eval")

In [ ]:
# Run this for a deeper score analysis on a question-by-question basis.
from process_sql import Schema, get_schema, get_sql
from evaluation import Evaluator
from exec_eval import eval_exec_match
import json
import csv

import nest_asyncio
nest_asyncio.apply()

predictions_path = f"{DATA_ROOT}/{MODEL['model']}/{PREDICTIONS_FILENAME}"
gold_path = f"{DATA_ROOT}/{MODEL['model']}/{GOLD_FILENAME}"
database_path = f"/kaggle/input/datasets/dcbisson23/test-suite-sql-eval-database/database"

with open(gold_path, "r") as f:
    # Took this code from evaluation.py.evaluate.
    glist = []
    gseq_one = []
    for l in f.readlines():
        if len(l.strip()) == 0:
            glist.append(gseq_one)
            gseq_one = []
        else:
            lstrip = l.strip().split('\t')
            gseq_one.append(lstrip)
    if len(gseq_one) != 0:
        glist.append(gseq_one)
    # End of block
# Not sure why but they nest glist in another list so I'm changing that here.
glist = glist[0]
evaluator = Evaluator()
results = []
with open(predictions_path, "r") as f:
    for idx, p_str in enumerate(f.readlines()):
        # Can't use question id for this since using gold
        g = glist[idx]
        # More taken from eval script.
        g_str, db_id = g
        db = os.path.join(database_path, db_id, db_id + ".sqlite")
        schema = Schema(get_schema(db))
        g_sql = get_sql(schema, g_str)
        hardness = evaluator.eval_hardness(g_sql)
        score = eval_exec_match(
            db=db,
            p_str=p_str,
            g_str=g_str,
            plug_value=False,
            keep_distinct=False,
            progress_bar_for_each_datapoint=False
        )
        result = {
            "id": idx,
            "db_id": db_id,
            "hardness": hardness,
            "score": score,
        }
        results.append(result)

results_path = f"{DATA_ROOT}/{MODEL['model']}/full_results.csv"
with open(results_path, "w", encoding="utf-8", newline="") as f_out:
    fieldnames = ["id", "db_id", "hardness", "score"]
    writer = csv.DictWriter(f_out, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)


In [ ]:
from scripts.utils import get_data_from_response
import json
import csv

responses_path = f"{DATA_ROOT}/{MODEL['model']}/{RESPONSES_FILENAME}"
gold_path = f"{DATA_ROOT}/{MODEL['model']}/{GOLD_FILENAME}"
results_path = f"{DATA_ROOT}/{MODEL['model']}/full_results.csv"

with open(QUESTIONS_PATH, "r") as f:
    questions = json.load(f)
with open(responses_path, "r") as f:
    responses = f.readlines()
with open(gold_path, "r") as f:
    golds = f.readlines()
with open(results_path, "r") as f:
    reader = csv.DictReader(f)
    results = [row for row in reader]


bad_items = []
for result in results:
    if int(result["score"]) == 0:
        response_json = responses[int(result["id"])]
        response_dict = json.loads(response_json)
        qid = response_dict["id"]
        question = questions[qid]["question"]
        response = get_data_from_response(response_dict["response"])["prediction"]
        bad_item = {
            "id": qid,
            "db_id": result["db_id"],
            "hardness": result["hardness"],
            "question": question,
            "prediction": predictions[int(result["id"])],
            "gold": golds[int(result["id"])]
        }
        bad_items.append(bad_item)
bad_results_path = f"{DATA_ROOT}/{MODEL['model']}/bad_predictions.csv"
with open(bad_results_path, "w", encoding="utf-8", newline="") as f_out:
    fieldnames = ["id", "db_id", "hardness", "question", "prediction", "gold"]
    writer = csv.DictWriter(f_out, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(bad_items)


In [3]:
from sentence_transformers import SentenceTransformer
from scripts.agents import AgentHarness, QueryTool, GetSchemaTool, TableSelectorTool, ExampleRetrieverTool

sentence_transformer = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 846.80it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
query_tool = QueryTool(db_root=SPIDER_DB_PATH)
# print(query_tool.forward(query="SELECT count(*) FROM singer", db_id="concert_singer"))

In [ ]:
schema_tool = GetSchemaTool(db_root=SPIDER_DB_PATH)
# print(schema_tool.forward(db_id="concert_singer"))

In [ ]:
table_tool = TableSelectorTool(db_root=SPIDER_DB_PATH, encoder=sentence_transformer)
# print(table_tool.forward(db_id="concert_singer", threshold=0.2, question="How many singers do we have?"))

In [4]:
example_tool = ExampleRetrieverTool(encoder=sentence_transformer, examples_path=EXAMPLES_PATH)
print(example_tool.forward(question="How many singers do we have?"))

['Question: How many artists are there?\nCosine Similarity: 0.7319285869598389\nQuery: SELECT count(*) FROM artist', 'Question: How many songs are there?\nCosine Similarity: 0.7355948090553284\nQuery: SELECT count(*) FROM Songs', 'Question: How many artists do we have?\nCosine Similarity: 0.8170273303985596\nQuery: SELECT count(*) FROM artist']


In [ ]:
from scripts.utils import load_gen_config, get_questions
import json
gen_config = load_gen_config(gen_config_path=f"{DATA_ROOT}/{MODEL['model']}/{GEN_CONFIG_FILENAME}")
agent = AgentHarness(
    model_id=MODEL["model"],
    gen_config=gen_config,
    tools=[query_tool, schema_tool, table_tool, example_tool]
)
questions = get_questions(questions_path=QUESTIONS_PATH, sample_rate=SAMPLE_RATE, offset=OFFSET)
with open(f"{DATA_ROOT}/{MODEL['model']}/{AGENT_RESULTS_FILENAME}", "w", encoding="utf8") as f_out:
    for question_dict in questions:
        result = agent.run(question=question_dict["question"], db_id=question_dict["db_id"], return_full_result=True)
        question_dict["result"] = result
        record = {
            "output": result.output,
            "state": result.state,
            "steps": result.steps,
            "token_usage": result.token_usage,
            "timing": result.timing
        }
        f_out.write(json.dumps(record) + "\n")